# Phase 10 — Clarification (Human-in-the-Loop)

> Run AFTER Phase 10 (restart kernel first). Cell 3 makes several Gemini calls and pauses mid-run — you'll resume it in cell 4.

**Goal:** an ambiguous query PAUSES the graph via `interrupt()`; resuming with a choice continues exactly where it stopped, yielding a scoped answer. Uses CLT-005, which holds both mega-cap tech (AAPL, MSFT) and high-beta tech (SNOW, CRM, ADBE) — a genuine ambiguity.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env'
print('env OK')

env OK


## 1. The ambiguity detector, standalone

In [2]:
from app.graph.clarifier import ClarifierNode
amb = ClarifierNode._detect('CLT-005', "How's my tech doing?")
print(amb)
clean = ClarifierNode._detect('CLT-002', 'What is my NVDA position worth?')
print(clean)

{"rows": 84, "clients": 10, "event": "portfolios_loaded", "level": "info", "timestamp": "2026-07-13T10:33:59.075288Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{'needs_clarification': True, 'question': "Your portfolio contains both mega-cap tech leaders and high-growth software companies. How would you like me to categorize 'tech' for this performance review?", 'options': ['Mega-cap tech (AAPL, MSFT, GOOGL, NVDA, META, AMZN)', 'High-growth/SaaS tech (CRM, ADBE, NFLX, SNOW)', 'All tech holdings combined', 'TSLA (often categorized separately as Automotive/Energy)']}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{'needs_clarification': False, 'question': '', 'options': []}


## 2. Config toggle — evals disable clarification so batch runs never block

In [3]:
from app.config import settings
print('ENABLE_CLARIFICATION default:', settings.ENABLE_CLARIFICATION)

ENABLE_CLARIFICATION default: True


## 3. Full graph run — watch it PAUSE

In [4]:
from langchain_core.messages import HumanMessage
from app.graph.builder import GraphBuilder
graph = GraphBuilder().with_all().build()
cfg = {'configurable': {'thread_id': 'CLT-005-nb10'}}
state = {'messages':[HumanMessage(content="How's my tech doing?")],
         'client_id':'CLT-005','session_id':'nb10'}
interrupt_payload = None
for chunk in graph.stream(state, cfg, stream_mode='updates'):
    if '__interrupt__' in chunk:
        interrupt_payload = chunk['__interrupt__'][0].value
        print('PAUSED. Interrupt payload:', interrupt_payload)
        break
    else:
        print('ran:', list(chunk.keys())[0])
print('\ngraph.get_state(cfg).next ->', graph.get_state(cfg).next, '(still paused)')

ran: memory_read
{"guard": "pii", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "input_guard", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:34:33.004670Z"}
{"guard": "prompt_injection", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "input_guard", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:34:33.005952Z"}
{"guard": "scope", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "input_guard", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:34:33.007211Z"}
ran: input_guard
{"query": "How's my tech doing?", "event": "planner_simple", "agent": "planner", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:34:33.008772Z"}


AFC is enabled with max remote calls: 10.


ran: planner


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"question": "Your portfolio contains both mega-cap tech leaders and high-growth software companies. How would you like me to categorize 'tech' for this performance review?", "options": ["Mega-cap tech (AAPL, MSFT, GOOGL, NVDA, META, AMZN)", "High-growth/SaaS tech (CRM, ADBE, NFLX, SNOW)", "All tech holdings combined", "TSLA (often categorized separately as Automotive/Energy)"], "event": "clarification_needed", "agent": "clarifier", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:34:34.166429Z"}
PAUSED. Interrupt payload: {'question': "Your portfolio contains both mega-cap tech leaders and high-growth software companies. How would you like me to categorize 'tech' for this performance review?", 'options': ['Mega-cap tech (AAPL, MSFT, GOOGL, NVDA, META, AMZN)', 'High-growth/SaaS tech (CRM, ADBE, NFLX, SNOW)', 'All tech holdings combined', 'TSLA (often categorized separately as Automotive/Energy)']}

graph.get_state(cfg).next -> ('clarifier',) (s

## 4. Resume with a choice — execution continues exactly where it stopped

In [5]:
from langgraph.types import Command
from app.agents.base import _text
resumed = graph.invoke(Command(resume='mega-cap only'), cfg)
answer = next(_text(m.content) for m in reversed(resumed['messages'])
              if getattr(m,'type',None)=='ai' and m.content)
print(answer)
print('\ngraph.get_state(cfg).next ->', graph.get_state(cfg).next, '(finished)')

AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"question": "Your portfolio contains both mega-cap tech leaders and high-growth software companies. How would you like me to categorize 'tech' for this performance review?", "options": ["Mega-cap tech (AAPL, MSFT, GOOGL, NVDA, META, AMZN)", "High-growth/SaaS tech (CRM, ADBE, NFLX, SNOW)", "All tech holdings combined", "TSLA (often categorized separately as Automotive/Energy)"], "event": "clarification_needed", "agent": "clarifier", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:34:41.631792Z"}
{"answer": "mega-cap only", "event": "clarification_resumed", "agent": "clarifier", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:34:41.632842Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"strategy": "llm", "decision": "portfolio", "event": "route_decision", "agent": "supervisor", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:34:48.914989Z"}
{"next_agent": "portfolio", "hop": 1, "event": "supervisor_dispatch", "agent": "supervisor", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:34:48.915836Z"}
{"query": "(Clarification) mega-cap only", "event": "agent_start", "agent": "portfolio", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:34:48.917714Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


...

{"chars": 1784, "revision": 1, "had_critique": true, "event": "synthesized", "agent": "synthesizer", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:36:23.634471Z"}
{"guard": "numeric_consistency", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "reflector", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:36:23.638370Z"}
{"guard": "conflict_disclosure", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "reflector", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:36:23.639992Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"


{"guard": "citation_coverage", "action": "revise", "passed": false, "reason": "unsupported claims: ['META current price ($669.21), return (597.68%), and absolute gain ($103,192.20)', 'AMZN current pr", "event": "guardrail_check", "agent": "reflector", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:36:41.089963Z"}
{"attempt": 2, "guard": "citation_coverage", "reason": "unsupported claims: ['META current price ($669.21), return (597.68%), and absolute gain ($103,192.20)', 'AMZN current pr", "event": "reflection_revise", "agent": "reflector", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:36:41.090556Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"


{"chars": 4495, "revision": 2, "had_critique": true, "event": "synthesized", "agent": "synthesizer", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:37:22.449426Z"}
{"guard": "numeric_consistency", "action": "revise", "passed": false, "reason": "answer contains numbers not found in tool evidence: [1174172.19, 948848.55, 172163.64, 268022.0, 232167.0, 154040.0, 120", "event": "guardrail_check", "agent": "reflector", "session_id": "nb10", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-13T10:37:22.454348Z"}
{"reason": "answer contains numbers not found in tool evidence: [1174172.19, 948848.55, 172163.64, 268022.0, 232167.0, 154040.0, 120", "event": "reflection_max_revisions", "agent": "reflector", "session_id": "nb10", "client_id": "CLT-005", "level": "warning", "timestamp": "2026-07-13T10:37:22.455689Z"}
{"client_id": "CLT-005", "session_id": "nb10", "event": "decision_saved", "level": "info", "timestamp": "2026-07-13T10:37:22.46

## ✅ Acceptance check

In [6]:
assert interrupt_payload is not None, 'ambiguous query should have paused the graph'
assert 'tech' in interrupt_payload['question'].lower() or 'mega' in str(interrupt_payload).lower()
assert answer
print('Phase 10 acceptance: PASS — interrupted, resumed, scoped answer produced')

Phase 10 acceptance: PASS — interrupted, resumed, scoped answer produced
